In [ ]:
import numpy as np
import torch
from scipy.ndimage import distance_transform_edt
import math

# --- 1. Load the Best Model ---
print("Loading Best Model for Final Evaluation...")
model = UNet3D(in_channels=4, out_channels=1).to(DEVICE)
model.load_state_dict(torch.load("best_model_m3.pth"))
model.eval()

# --- 2. Helper Functions (pFM & PSNR) ---
def calculate_pfm(pred_bin, gt_bin):
    """Pseudo F-Measure (Distance-Based)"""
    if gt_bin.sum() == 0: return 0.0 if pred_bin.sum() > 0 else 1.0
    dist_gt = distance_transform_edt(1 - gt_bin)
    dist_pred = distance_transform_edt(1 - pred_bin)
    
    recall_w = 1.0 / (1.0 + dist_pred)
    pseudo_rec = np.sum(recall_w[gt_bin]) / np.sum(gt_bin)
    
    prec_w = 1.0 / (1.0 + dist_gt)
    if np.sum(pred_bin) == 0: pseudo_prec = 0.0
    else: pseudo_prec = np.sum(prec_w[pred_bin]) / np.sum(pred_bin)
    
    if (pseudo_rec + pseudo_prec) == 0: return 0.0
    return 2 * pseudo_rec * pseudo_prec / (pseudo_rec + pseudo_prec)

# --- 3. Run Inference with TTA (Test-Time Augmentation) ---
print("Running TTA Inference (Original + HFlip + VFlip)...")

best_threshold = 0.40 # Found via validation
metrics = {'tp': 0, 'fp': 0, 'fn': 0, 'pfm': 0, 'psnr': 0, 'count': 0}

with torch.no_grad():
    for images, masks in tqdm(val_loader, desc="TTA Eval"):
        images = images.to(DEVICE)
        
        # TTA: Forward Pass 3x
        # 1. Original
        p1 = torch.sigmoid(model(images)[:, :, CENTER_SLICE_IDX, :, :])
        # 2. Horizontal Flip
        p2 = torch.flip(torch.sigmoid(model(torch.flip(images, [3]))[:, :, CENTER_SLICE_IDX, :, :]), [3])
        # 3. Vertical Flip
        p3 = torch.flip(torch.sigmoid(model(torch.flip(images, [2]))[:, :, CENTER_SLICE_IDX, :, :]), [2])
        
        # Average Predictions
        probs = (p1 + p2 + p3) / 3.0
        preds_bin = (probs > best_threshold).float().cpu()
        masks = masks.cpu()

        # Update Counts
        metrics['tp'] += (preds_bin * masks).sum().item()
        metrics['fp'] += (preds_bin * (1 - masks)).sum().item()
        metrics['fn'] += ((1 - preds_bin) * masks).sum().item()
        
        # Pixel-wise Metrics
        for i in range(images.shape[0]):
            metrics['pfm'] += calculate_pfm(preds_bin[i].numpy(), masks[i].numpy())
            mse = np.mean((preds_bin[i].numpy() - masks[i].numpy()) ** 2)
            metrics['psnr'] += 100 if mse == 0 else 20 * math.log10(1.0 / math.sqrt(mse))
            metrics['count'] += 1

# --- 4. Print Report ---
recall = metrics['tp'] / (metrics['tp'] + metrics['fn'] + 1e-8)
precision = metrics['tp'] / (metrics['tp'] + metrics['fp'] + 1e-8)
f05 = (1.25 * metrics['tp']) / (1.25 * metrics['tp'] + 0.25 * metrics['fn'] + metrics['fp'] + 1e-8)

print("-" * 40)
print(f"🏆 FINAL M3 RESULTS (TTA Enabled)")
print("-" * 40)
print(f"F0.5 Score:  {f05:.4f}")
print(f"Precision:   {precision:.4f}")
print(f"Recall:      {recall:.4f}")
print(f"pFM Score:   {metrics['pfm']/metrics['count']:.4f}")
print(f"PSNR:        {metrics['psnr']/metrics['count']:.2f} dB")
print("-" * 40)

In [ ]:
# --- EXPERIMENTAL: Failed Z-Roll Augmentation ---
# This class demonstrates the "Robustness Insight" from the report.
# Enabling 'do_z_roll=True' causes F0.5 to drop < 0.20 because
# the 2D labels become misaligned with the 3D volume.
class GeometricInkDataset_ZRoll(GeometricInkDataset):
    def __getitem__(self, idx):
        # ... standard load ...
        y, x = self.valid_indices[idx]
        vol_patch = self.volume[:, y:y+self.patch_size, x:x+self.patch_size].astype(np.float32) / 255.0
        
        # THE FAILED AUGMENTATION:
        # Randomly shift the volume up/down by 0-2 slices
        shift = np.random.randint(-2, 3) 
        vol_patch = np.roll(vol_patch, shift, axis=0) 
        
        # ... rest of pipeline ...
        # (This breaks center-slice supervision)
        return super().__getitem__(idx) # simplified for display

In [ ]:
# --- EXPERIMENTAL: Failed Z-Roll Augmentation ---
# This class demonstrates the "Robustness Insight" from the report.
# Enabling 'do_z_roll=True' causes F0.5 to drop < 0.20 because
# the 2D labels become misaligned with the 3D volume.
class GeometricInkDataset_ZRoll(GeometricInkDataset):
    def __getitem__(self, idx):
        # ... standard load ...
        y, x = self.valid_indices[idx]
        vol_patch = self.volume[:, y:y+self.patch_size, x:x+self.patch_size].astype(np.float32) / 255.0
        
        # THE FAILED AUGMENTATION:
        # Randomly shift the volume up/down by 0-2 slices
        shift = np.random.randint(-2, 3) 
        vol_patch = np.roll(vol_patch, shift, axis=0) 
        
        # ... rest of pipeline ...
        # (This breaks center-slice supervision)
        return super().__getitem__(idx) # simplified for display

In [ ]:
import matplotlib.pyplot as plt

# Get a batch
images, masks = next(iter(val_loader))
images = images.to(DEVICE)
with torch.no_grad():
    preds = torch.sigmoid(model(images)[:, :, CENTER_SLICE_IDX, :, :])

# Plot top 3
plt.figure(figsize=(12, 8))
for i in range(3):
    plt.subplot(3, 4, i*4 + 1)
    plt.title("Input (Intensity)")
    plt.imshow(images[i, 0, 2, :, :].cpu(), cmap='gray')
    
    plt.subplot(3, 4, i*4 + 2)
    plt.title("Input (Gradient Z)")
    plt.imshow(images[i, 1, 2, :, :].cpu(), cmap='jet') # Show the geometric feature!
    
    plt.subplot(3, 4, i*4 + 3)
    plt.title("Prediction")
    plt.imshow(preds[i].cpu(), cmap='gray')
    
    plt.subplot(3, 4, i*4 + 4)
    plt.title("Ground Truth")
    plt.imshow(masks[i].cpu(), cmap='gray')
plt.tight_layout()
plt.show()